# 🚀 Модуль 10. Урок 3. Structured Outputs через Pydantic: гарантия схемы и типизации

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_10_llm_api/lesson_03_structured_outputs_pydantic_json.ipynb)

## 📌 Бизнес-контекст и методология
В продакшене аналитик не имеет права получать от модели произвольный текст.
Использование Pydantic позволяет валидировать входные и выходные типы данных, гарантируя соблюдение схемы перед записью в хранилище.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
from pydantic import BaseModel, Field, ValidationError
from typing import List, Literal

class SentimentAnalysisOutput(BaseModel):
    sentiment: Literal['POSITIVE', 'NEUTRAL', 'NEGATIVE']
    score: float = Field(..., ge=0.0, le=1.0, description="Уверенность модели")
    detected_topics: List[str] = Field(default_factory=list)

# Проверка валидного ответа
valid_payload = {
    "sentiment": "NEGATIVE",
    "score": 0.94,
    "detected_topics": ["slow_delivery", "damaged_box"]
}

parsed_obj = SentimentAnalysisOutput(**valid_payload)
print("🟢 Успешная валидация объекта Pydantic:")
print(parsed_obj.model_dump())


🟢 Успешная валидация объекта Pydantic:
{'sentiment': 'NEGATIVE', 'score': 0.94, 'detected_topics': ['slow_delivery', 'damaged_box']}


### 🎯 Практическое задание 1: Перехват невалидных данных
Проверьте, что передача `score = 1.5` (вне диапазона [0, 1]) вызывает исключение `ValidationError`.

In [3]:
### ВАШ КОД ЗДЕСЬ
caught_error = False
try:
    bad_payload = {"sentiment": "POSITIVE", "score": 1.5, "detected_topics": []}
    SentimentAnalysisOutput(**bad_payload)
except ValidationError:
    caught_error = True

assert caught_error, "Pydantic обязан отловить нарушение валидации диапазона score"
print("✅ Тест успешно пройден: строгий контракт схемы защищает пайплайн!")


✅ Тест успешно пройден: строгий контракт схемы защищает пайплайн!


### 💡 Интеграция с базой данных
Объекты Pydantic конвертируются в словари через `.model_dump()` и напрямую вставляются в PostgreSQL через SQLAlchemy или DuckDB.